In [1]:
from datasets import load_dataset

ds = load_dataset("jxm/mpqa")

In [2]:
# -----------------------------
# Imports
# -----------------------------
import numpy as np
import pandas as pd
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, RandomSampler, SequentialSampler
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import warnings
warnings.filterwarnings('ignore')

nltk.download('stopwords')
nltk.download('wordnet')

# -----------------------------
# Load dataset (example MPQA)
# -----------------------------
from datasets import load_dataset
ds = load_dataset("jxm/mpqa")
df = ds["train"].to_pandas()
df = df.dropna(subset=['sentence', 'label'])

# -----------------------------
# Data Cleaning
# -----------------------------
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return ' '.join(words)

df['text'] = df['sentence'].apply(clean_text)

# Label encoding
label_encoder = LabelEncoder()
df['label_enc'] = label_encoder.fit_transform(df['label'])

# Remove duplicates
df.drop_duplicates(inplace=True)
df = df.reset_index(drop=True)

# -----------------------------
# Tokenization
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-base')

def tokenize_text(texts, max_len=36):
    encodings = tokenizer(
        texts.tolist(),
        padding='max_length',
        truncation=True,
        max_length=max_len,
        return_tensors='pt'
    )
    return encodings['input_ids'], encodings['attention_mask']

input_ids, attention_masks = tokenize_text(df['text'])
labels = torch.tensor(df['label_enc'].values)

# -----------------------------
# Train/Val/Test Split
# -----------------------------
train_idx, temp_idx = train_test_split(np.arange(len(labels)), test_size=0.2, stratify=labels, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=labels[temp_idx], random_state=42)

train_dataset = TensorDataset(input_ids[train_idx], attention_masks[train_idx], labels[train_idx])
val_dataset = TensorDataset(input_ids[val_idx], attention_masks[val_idx], labels[val_idx])
test_dataset = TensorDataset(input_ids[test_idx], attention_masks[test_idx], labels[test_idx])

batch_size = 32
train_loader = DataLoader(train_dataset, sampler=RandomSampler(train_dataset), batch_size=batch_size)
val_loader = DataLoader(val_dataset, sampler=SequentialSampler(val_dataset), batch_size=batch_size)
test_loader = DataLoader(test_dataset, sampler=SequentialSampler(test_dataset), batch_size=batch_size)

# -----------------------------
# DeBERTa-BiLSTM Model
# -----------------------------
class DeBERTa_BiLSTM(nn.Module):
    def __init__(self, num_classes, hidden_size=128, num_layers=2, bidirectional=True):
        super(DeBERTa_BiLSTM, self).__init__()
        self.deberta = AutoModel.from_pretrained('microsoft/deberta-base')
        self.lstm = nn.LSTM(
            input_size=768,  # DeBERTa base hidden size
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bidirectional
        )
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(hidden_size * 2 if bidirectional else hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        # Freeze DeBERTa embeddings
        with torch.no_grad():
            deberta_outputs = self.deberta(input_ids=input_ids, attention_mask=attention_mask)
        x = deberta_outputs.last_hidden_state.float()  # cast to float32 for LSTM
        x, _ = self.lstm(x)
        x = self.dropout(x[:, -1, :])
        x = self.fc(x)
        return x

# -----------------------------
# Training Setup
# -----------------------------
device = 'cuda' if torch.cuda.is_available() else 'cpu'
num_classes = len(label_encoder.classes_)

model = DeBERTa_BiLSTM(num_classes=num_classes).to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)
loss_fn = nn.CrossEntropyLoss()

# -----------------------------
# Training & Evaluation Functions
# -----------------------------
def train_epoch(model, dataloader, optimizer, loss_fn, device):
    model.train()
    total_loss, total_acc = 0, 0
    for batch in dataloader:
        input_ids, mask, labels = [b.to(device) for b in batch]
        optimizer.zero_grad()
        outputs = model(input_ids, mask)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_acc += (outputs.argmax(1) == labels).sum().item() / labels.size(0)
    return total_loss / len(dataloader), total_acc / len(dataloader)

def eval_model(model, dataloader, loss_fn, device):
    model.eval()
    total_loss, total_acc = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            input_ids, mask, labels = [b.to(device) for b in batch]
            outputs = model(input_ids, mask)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()
            preds = outputs.argmax(1)
            total_acc += (preds == labels).sum().item() / labels.size(0)
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())
    return total_loss / len(dataloader), total_acc / len(dataloader), torch.cat(all_preds), torch.cat(all_labels)

# -----------------------------
# Training Loop
# -----------------------------
epochs = 10
best_acc = 0
for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, loss_fn, device)
    val_loss, val_acc, val_preds, val_labels = eval_model(model, val_loader, loss_fn, device)

    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.3f}, Train Acc: {train_acc:.3f}")
    print(f"Val Loss: {val_loss:.3f}, Val Acc: {val_acc:.3f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

DebertaModel LOAD REPORT from: microsoft/deberta-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1/10
Train Loss: 0.589, Train Acc: 0.749
Val Loss: 0.561, Val Acc: 0.754
Epoch 2/10
Train Loss: 0.560, Train Acc: 0.749
Val Loss: 0.557, Val Acc: 0.754
Epoch 3/10
Train Loss: 0.555, Train Acc: 0.751
Val Loss: 0.551, Val Acc: 0.754
Epoch 4/10
Train Loss: 0.547, Train Acc: 0.751
Val Loss: 0.547, Val Acc: 0.754
Epoch 5/10
Train Loss: 0.542, Train Acc: 0.748
Val Loss: 0.543, Val Acc: 0.756
Epoch 6/10
Train Loss: 0.532, Train Acc: 0.751
Val Loss: 0.543, Val Acc: 0.760
Epoch 7/10
Train Loss: 0.530, Train Acc: 0.751
Val Loss: 0.544, Val Acc: 0.763
Epoch 8/10
Train Loss: 0.524, Train Acc: 0.752
Val Loss: 0.540, Val Acc: 0.761
Epoch 9/10
Train Loss: 0.524, Train Acc: 0.752
Val Loss: 0.535, Val Acc: 0.767
Epoch 10/10
Train Loss: 0.521, Train Acc: 0.752
Val Loss: 0.530, Val Acc: 0.763


In [3]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score,accuracy_score

# -----------------------------
# Test Evaluation with Metrics
# -----------------------------
model.load_state_dict(torch.load("best_model.pth"))
_, test_acc, test_preds, test_labels = eval_model(model, test_loader, loss_fn, device)

# -----------------------------
# Test Evaluation with Metrics
# -----------------------------
model.load_state_dict(torch.load("best_model.pth"))
_, test_acc, test_preds, test_labels = eval_model(model, test_loader, loss_fn, device)

# Convert tensors to numpy
y_true = test_labels.numpy()
y_pred = test_preds.numpy()

# Accuracy
accuracy = accuracy_score(y_true, y_pred)

# Precision, Recall, F1 (macro)
precision = precision_score(y_true, y_pred, average='macro')
recall = recall_score(y_true, y_pred, average='macro')
f1 = f1_score(y_true, y_pred, average='macro')

# Specificity for binary
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)

# ROC-AUC (binary)
probs_list = []
with torch.no_grad():
    for batch in test_loader:
        input_ids, mask, labels = [b.to(device) for b in batch]
        logits = model(input_ids, mask)
        probs_list.append(F.softmax(logits, dim=1)[:, 1].cpu())  # probability of positive class
probs = torch.cat(probs_list).numpy()
roc_auc = roc_auc_score(y_true, probs)

# Print all metrics
print(f"Accuracy      : {accuracy:.4f}")
print(f"Precision     : {precision:.4f}")
print(f"Recall        : {recall:.4f}")
print(f"Specificity   : {specificity:.4f}")
print(f"F1 Score      : {f1:.4f}")
print(f"ROC-AUC       : {roc_auc:.4f}")

Accuracy      : 0.7621
Precision     : 0.6959
Recall        : 0.5535
Specificity   : 0.9725
F1 Score      : 0.5404
ROC-AUC       : 0.6978


In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import transformers
from transformers import AutoModel, AutoTokenizer
from datasets import load_dataset
import nltk
from nltk.corpus import stopwords
import re
import warnings
from torch.utils.data import DataLoader, TensorDataset, RandomSampler, SequentialSampler
import time
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import ParameterSampler
import random

warnings.filterwarnings('ignore')

# Load dataset
ds = load_dataset("jxm/mpqa")
df = ds["train"].to_pandas()

def preprocess_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = text.lower()
    return text

df['cleaned_text'] = df['sentence'].apply(preprocess_text)
df = df[df['cleaned_text'] != '[deleted]']
df = df[df['cleaned_text'] != '[removed]']

# Download NLTK stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_statement(statement):
    statement = statement.lower()
    statement = re.sub(r'[^\w\s]', '', statement)
    statement = re.sub(r'\d+', '', statement)
    words = statement.split()
    words = [word for word in words if word not in stop_words]
    return ' '.join(words)

df['cleaned_text'] = df['cleaned_text'].apply(clean_statement)

# Label encoding
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['label'])

# Balance dataset
class_counts = df['label'].value_counts()
print("Class distribution before balancing:")
print(class_counts)

majority_class = df['label'].value_counts().idxmax()
minority_class = df['label'].value_counts().idxmin()
df_majority = df[df['label'] == majority_class]
df_minority = df[df['label'] == minority_class]
df_majority_downsampled = df_majority.sample(len(df_minority), random_state=42)
df_balanced = pd.concat([df_majority_downsampled, df_minority])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nClass distribution after balancing:")
print(df_balanced['label'].value_counts())

# Tokenization
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")
encoded_texts = tokenizer(
    df_balanced['cleaned_text'].tolist(),
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encoded_texts['input_ids']
attention_masks = encoded_texts['attention_mask']
labels = torch.tensor(df_balanced['label'].values, dtype=torch.long)

# Train-validation split
train_idx, val_idx = train_test_split(
    range(len(labels)), test_size=0.2, random_state=42, stratify=labels
)

train_inputs = input_ids[train_idx]
val_inputs = input_ids[val_idx]
train_masks = attention_masks[train_idx]
val_masks = attention_masks[val_idx]
train_labels = labels[train_idx]
val_labels = labels[val_idx]

train_data = TensorDataset(train_inputs, train_masks, train_labels)
val_data = TensorDataset(val_inputs, val_masks, val_labels)

class DeBERTaBiLSTM(nn.Module):
    def __init__(self, num_classes=2):
        super(DeBERTaBiLSTM, self).__init__()
        self.bert = AutoModel.from_pretrained(
            "microsoft/deberta-v3-base", 
            torch_dtype=torch.float32
        )
        self.bilstm = nn.LSTM(
            input_size=768,
            hidden_size=128,
            batch_first=True,
            bidirectional=True,
            dropout=0.1
        )
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(256, num_classes)

    def forward(self, input_ids, attention_mask):
        input_ids = input_ids.to(torch.long)
        attention_mask = attention_mask.to(torch.long)
        
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        lstm_input = bert_output.last_hidden_state.float()
        lstm_output, _ = self.bilstm(lstm_input)
        last_hidden = lstm_output[:, -1, :]
        last_hidden = self.dropout(last_hidden)
        output = self.fc(last_hidden)
        return output

# Initialize model
model = DeBERTaBiLSTM()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

print(f"Using device: {device}")
print(f"Train samples: {len(train_data)}, Val samples: {len(val_data)}")

# Hyperparameter search
param_grid = {
    'learning_rate': [2e-5, 3e-5],
    'batch_size': [16, 32],
    'epochs': [2]
}

num_samples = 2
param_list = list(ParameterSampler(param_grid, n_iter=num_samples, random_state=42))

best_model_state = None
best_val_accuracy = 0.0
best_params = None
best_all_preds = None
best_all_labels = None

for idx, params in enumerate(param_list):
    print(f"\n{'='*60}")
    print(f"Testing configuration {idx + 1}/{num_samples}: {params}")
    print(f"{'='*60}")
    
    learning_rate = params['learning_rate']
    batch_size = params['batch_size']
    epochs = params['epochs']
    
    train_dataloader = DataLoader(train_data, sampler=RandomSampler(train_data), batch_size=batch_size)
    val_dataloader = DataLoader(val_data, sampler=SequentialSampler(val_data), batch_size=batch_size)
    
    optimizer = AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
    total_steps = len(train_dataloader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)
    
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        total_train_accuracy = 0
        
        for step, batch in enumerate(train_dataloader):
            b_input_ids, b_input_mask, b_labels = tuple(t.to(device) for t in batch)
            
            optimizer.zero_grad()
            outputs = model(b_input_ids, b_input_mask)
            loss = nn.CrossEntropyLoss()(outputs, b_labels)
            total_train_loss += loss.item()
            
            logits = outputs.detach().cpu().numpy()
            label_ids = b_labels.cpu().numpy()
            total_train_accuracy += flat_accuracy(logits, label_ids)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
        
        avg_train_loss = total_train_loss / len(train_dataloader)
        avg_train_accuracy = total_train_accuracy / len(train_dataloader)
        
        # Validation
        model.eval()
        total_val_loss = 0
        total_val_accuracy = 0
        all_preds_val = []
        all_labels_val = []
        all_probs_val = []
        
        for batch in val_dataloader:
            b_input_ids, b_input_mask, b_labels = tuple(t.to(device) for t in batch)
            
            with torch.no_grad():
                outputs = model(b_input_ids, b_input_mask)
                loss = nn.CrossEntropyLoss()(outputs, b_labels)
                total_val_loss += loss.item()
                
                logits = outputs.detach().cpu().numpy()
                probs = torch.nn.functional.softmax(torch.tensor(logits), dim=1).numpy()
                label_ids = b_labels.cpu().numpy()
                total_val_accuracy += flat_accuracy(logits, label_ids)
                
                all_preds_val.extend(np.argmax(logits, axis=1))
                all_labels_val.extend(label_ids)
                all_probs_val.extend(probs)
        
        avg_val_loss = total_val_loss / len(val_dataloader)
        avg_val_accuracy = total_val_accuracy / len(val_dataloader)
        
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_accuracy:.4f}")
        print(f"Val Loss: {avg_val_loss:.4f} | Val Acc: {avg_val_accuracy:.4f}")
        
        if avg_val_accuracy > best_val_accuracy:
            best_val_accuracy = avg_val_accuracy
            best_model_state = model.state_dict().copy()
            best_params = params.copy()
            best_all_preds = np.array(all_preds_val)
            best_all_labels = np.array(all_labels_val)
            best_all_probs = np.array(all_probs_val)
            print(f"  → New best model! Val Acc: {best_val_accuracy:.4f}")

print(f"\n{'='*60}")
print(f"BEST RESULTS: Val Accuracy = {best_val_accuracy:.4f}")
print(f"Best Parameters: {best_params}")
print(f"{'='*60}")

# Load best model and compute ALL METRICS
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    
    # Compute metrics
    y_true = best_all_labels
    y_pred = best_all_preds
    y_proba = best_all_probs
    
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    roc_auc = roc_auc_score(y_true, y_proba[:, 1], multi_class='ovr')
    
    # Specificity calculation
    cm = confusion_matrix(y_true, y_pred)
    specificity = cm[0, 0] / (cm[0, 0] + cm[0, 1]) if (cm[0, 0] + cm[0, 1]) > 0 else 0.0
    
    # 🎯 YOUR EXACT REQUESTED OUTPUT FORMAT
    print("\n" + "="*50)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall : {recall:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"ROC-AUC : {roc_auc:.4f}")
    print("="*50)
    
    # Classification report with fixed class names
    class_names = [str(cls) for cls in label_encoder.classes_]
    print("\nFinal Classification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names))
    
    torch.save(model.state_dict(), 'deberta_bilstm_best.pth')
    print("\n✅ Model saved as 'deberta_bilstm_best.pth'")
else:
    print("No best model found.")

README.md:   0%|          | 0.00/650 [00:00<?, ?B/s]

data/train-00000-of-00001-a7df005a1b0788(…):   0%|          | 0.00/141k [00:00<?, ?B/s]

data/test-00000-of-00001-05fc5ca1c399669(…):   0%|          | 0.00/33.7k [00:00<?, ?B/s]

data/dev-00000-of-00001-8814a3252cc44468(…):   0%|          | 0.00/6.19k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8603 [00:00<?, ? examples/s]

...

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Using device: cuda
Train samples: 3697, Val samples: 925

Testing configuration 1/2: {'learning_rate': 3e-05, 'epochs': 2, 'batch_size': 16}
Epoch 1/2 | Train Loss: 0.5696 | Train Acc: 0.7220
Val Loss: 0.5378 | Val Acc: 0.7937
  → New best model! Val Acc: 0.7937
Epoch 2/2 | Train Loss: 0.4375 | Train Acc: 0.8246
Val Loss: 0.4777 | Val Acc: 0.8142
  → New best model! Val Acc: 0.8142

Testing configuration 2/2: {'learning_rate': 3e-05, 'epochs': 2, 'batch_size': 32}
Epoch 1/2 | Train Loss: 0.4591 | Train Acc: 0.8194
Val Loss: 0.4589 | Val Acc: 0.8142
  → New best model! Val Acc: 0.8142
Epoch 2/2 | Train Loss: 0.3739 | Train Acc: 0.8637
Val Loss: 0.4574 | Val Acc: 0.8305
  → New best model! Val Acc: 0.8305

BEST RESULTS: Val Accuracy = 0.8305
Best Parameters: {'learning_rate': 3e-05, 'epochs': 2, 'batch_size': 32}

Accuracy : 0.8303
Precision : 0.8312
Recall : 0.8303
Specificity: 0.8035
F1 Score : 0.8302
ROC-AUC : 0.8910

Final Classification Report:
              precision    recall  f1-

In [2]:
from datasets import load_dataset

ds = load_dataset("mattymchen/mr")

README.md:   0%|          | 0.00/688 [00:00<?, ?B/s]

data/test-00000-of-00001-1ad570418120a67(…):   0%|          | 0.00/884k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/10662 [00:00<?, ? examples/s]

In [2]:
from datasets import load_dataset
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import DebertaTokenizer, DebertaModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)
import numpy as np

# ===================
# 1. Load data
# ===================
ds = load_dataset("mattymchen/mr", split="test")  # or use "train" for full training
df = ds.to_pandas()
texts = df["text"].reset_index(drop=True)
labels = df["label"].reset_index(drop=True)

# ===================
# 2. Dataset class
# ===================
class MRDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):   # shorter context
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])
        label = int(self.labels.iloc[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label": torch.tensor(label, dtype=torch.long),
        }


# ===================
# 3. Tokenizer & splits
# ===================
tokenizer = DebertaTokenizer.from_pretrained("microsoft/deberta-base")

texts_train, texts_test, labels_train, labels_test = train_test_split(
    texts, labels, test_size=0.2, stratify=labels, random_state=42
)

train_dataset = MRDataset(texts_train, labels_train, tokenizer)
test_dataset  = MRDataset(texts_test,  labels_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)    # smaller batch
test_loader  = DataLoader(test_dataset,  batch_size=8, shuffle=False)

# ===================
# 4. Clearer WEAKER model (much smaller + high dropout)
# ===================
class DebertaBiLSTMClassifier(nn.Module):
    def __init__(self, deberta_name="microsoft/deberta-base", hidden_size=64, num_classes=2, dropout=0.8):
        super().__init__()
        self.deberta = DebertaModel.from_pretrained(deberta_name)
        self.deberta_hidden = self.deberta.config.hidden_size  # 768

        # Much smaller BiLSTM + very high dropout
        self.bilstm = nn.LSTM(
            input_size=self.deberta_hidden,
            hidden_size=hidden_size,           # 64
            num_layers=1,
            bidirectional=True,
            batch_first=True,
            dropout=dropout,                   # 80%
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size * 2, num_classes)  # 128 → 2

    def forward(self, input_ids, attention_mask):
        outputs = self.deberta(input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state  # (B, L, H)

        lstm_out, _ = self.bilstm(sequence_output)  # (B, L, 128)

        # Mean pooling over sequence (same as before)
        lstm_pooled = lstm_out.mean(dim=1)  # (B, 128)

        logits = self.classifier(self.dropout(lstm_pooled))
        return logits


model = DebertaBiLSTMClassifier(num_classes=2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
criterion = nn.CrossEntropyLoss()

# ===================
# 5. Train loop (only 1 epoch, even less training)
# ===================
model.train()
epochs = 5  # single epoch => underfit
for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    total_loss = 0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"  Train Loss: {avg_loss:.4f}")

# ===================
# 6. Test & metrics
# ===================
model.eval()
all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].cpu().numpy()

        logits = model(input_ids, attention_mask)      # (B, 2)
        probs = torch.softmax(logits, dim=-1)          # (B, 2)
        pred  = torch.argmax(logits, dim=-1)           # (B,)

        all_labels.extend(labels)
        all_preds.extend(pred.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

labels = np.array(all_labels)
preds  = np.array(all_preds)
probs  = np.array(all_probs)

acc  = accuracy_score(labels, preds)
prec = precision_score(labels, preds, pos_label=1, zero_division=0)
rec  = recall_score(labels, preds, pos_label=1, zero_division=0)
f1   = f1_score(labels, preds, pos_label=1, zero_division=0)

# Specificity
tn = ((labels == 0) & (preds == 0)).sum()
fp = ((labels == 0) & (preds == 1)).sum()
spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

auc = roc_auc_score(labels, probs) if len(set(labels)) > 1 else 0.0

print("\nTest Metrics:")
print(f"Accuracy:    {acc:.4f}")
print(f"Precision:   {prec:.4f}")
print(f"Recall:      {rec:.4f}")
print(f"Specificity: {spec:.4f}")
print(f"F1:          {f1:.4f}")
print(f"AUC:         {auc:.4f}")

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

DebertaModel LOAD REPORT from: microsoft/deberta-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.8 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Epoch 1/5
  Train Loss: 0.6778
Epoch 2/5
  Train Loss: 0.4739
Epoch 3/5
  Train Loss: 0.3164
Epoch 4/5
  Train Loss: 0.2214
Epoch 5/5
  Train Loss: 0.1557

Test Metrics:
Accuracy:    0.8322
Precision:   0.7702
Recall:      0.9465
Specificity: 0.7179
F1:          0.8493
AUC:         0.9192
